In this notebook, we load models trained with best hyperparameters and analyze their performance.

In [1]:
import json
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Device: {device}")

import keras
import tensorflow as tf

from load_dataset import build_dataset, make_tf_dataset, make_torch_dataset
from model_swin_unetr import build_swin_unetr_mc

import metric_tf
import metric_torch

print(f"Tensorflow devices: {tf.config.list_physical_devices()}")

PyTorch Device: cuda


2026-04-30 10:26:17.581057: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-30 10:26:17.906661: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-30 10:26:18.632978: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


Tensorflow devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


2026-04-30 10:26:21.311474: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-30 10:26:21.312001: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


# Load trained models

In [2]:
unet_model_path = "output_unet3d_56.keras"
swin_model_path = "output_swin_unetr_56.pth"

unet_model = keras.models.load_model(unet_model_path, compile = False)
unet_model.compile(
    optimizer = keras.optimizers.Adam(),
    loss = metric_tf.bce_dice_loss,
    metrics=["accuracy"]
)

swin_model = build_swin_unetr_mc()
swin_model_state_dict = torch.load(swin_model_path, weights_only = False, map_location = device)
swin_model.load_state_dict(swin_model_state_dict["model_state_dict"])
swin_model = swin_model.to(device)

# Load test dataset

In [3]:
x_test, y_test = build_dataset(dataset="test")

  Loading 1/38: BraTS-PED-00002-000
  Loading 2/38: BraTS-PED-00008-000
  Loading 3/38: BraTS-PED-00009-000
  Loading 4/38: BraTS-PED-00010-000
  Loading 5/38: BraTS-PED-00028-000
  Loading 6/38: BraTS-PED-00029-000
  Loading 7/38: BraTS-PED-00032-000
  Loading 8/38: BraTS-PED-00034-000
  Loading 9/38: BraTS-PED-00041-000
  Loading 10/38: BraTS-PED-00056-000
  Loading 11/38: BraTS-PED-00061-000
  Loading 12/38: BraTS-PED-00062-000
  Loading 13/38: BraTS-PED-00063-000
  Loading 14/38: BraTS-PED-00065-000
  Loading 15/38: BraTS-PED-00068-000
  Loading 16/38: BraTS-PED-00076-000
  Loading 17/38: BraTS-PED-00077-000
  Loading 18/38: BraTS-PED-00116-000
  Loading 19/38: BraTS-PED-00117-000
  Loading 20/38: BraTS-PED-00123-000
  Loading 21/38: BraTS-PED-00138-000
  Loading 22/38: BraTS-PED-00148-000
  Loading 23/38: BraTS-PED-00152-000
  Loading 24/38: BraTS-PED-00160-000
  Loading 25/38: BraTS-PED-00161-000
  Loading 26/38: BraTS-PED-00164-000
  Loading 27/38: BraTS-PED-00173-000
  Loading 

# U-Net Inference

In [4]:
print(f"Running U-Net inference")
test_tf_dataset = make_tf_dataset(x_test, y_test, training = False)
unet_predictions = unet_model.predict(test_tf_dataset)
print(f"Predictions shape: {unet_predictions.shape}")

Running U-Net inference
76/76 ━━━━━━━━━━━━━━━━━━━━ 11s 133ms/step
Predictions shape: (304, 64, 64, 64, 1)


# U-Net Metrics

In [12]:
import numpy as np

y_true_tf = tf.constant(y_test, dtype = tf.float32)
y_prediction_tf = tf.constant(unet_predictions, dtype = tf.float32)

unet_loss = metric_tf.bce_dice_loss(y_true_tf, y_prediction_tf).numpy()
unet_dice = metric_tf.dice_coef(y_true_tf, y_prediction_tf).numpy()
unet_hausdorffdistance = metric_torch.hausdorff_distance( # Squeeze below to remove the last dimension
    torch.from_numpy(y_test).squeeze(-1),
    torch.from_numpy(unet_predictions).squeeze(-1)
)

print(f"U-Net Loss: {unet_loss}")
print(f"U-Net Dice: {unet_dice}")
print(f"U-Net Hausdorff Distance: {unet_hausdorffdistance}")

batch_size = 4
all_means = []
all_stds = []

for i in range(0, len(x_test), batch_size):
    batch_x = x_test[i : i + batch_size]
    batch_mean, batch_std, _ = metric_tf.mc_prediction(unet_model, batch_x, num_passes = 10)
    all_means.append(batch_mean)
    all_stds.append(batch_std)
    print(f"Batch {i // batch_size + 1} done")

# U-net MC dropout, run mc_prediction for 50 times then calcutae the mean, std, and BCE+Dice loss
unet_mean = np.concatenate(all_means, axis = 0)
unet_std = np.concatenate(all_stds, axis = 0)

print(f"U-Net MC Uncertainty — Mean std: {unet_std.mean():.6f},  Max std: {unet_std.max():.6f}")
print(f"U-Net MC Loss (on mean prediction): {metric_tf.bce_dice_loss(tf.constant(y_test, dtype=tf.float32), tf.constant(unet_mean, dtype=tf.float32)).numpy():.4f}")

U-Net Loss: 0.6190009117126465
U-Net Dice: 0.7887554168701172
U-Net Hausdorff Distance: 23.66427224478046
Monte Carlo pass 1/10 completed
Monte Carlo pass 2/10 completed
Monte Carlo pass 3/10 completed
Monte Carlo pass 4/10 completed
Monte Carlo pass 5/10 completed
Monte Carlo pass 6/10 completed
Monte Carlo pass 7/10 completed
Monte Carlo pass 8/10 completed
Monte Carlo pass 9/10 completed
Monte Carlo pass 10/10 completed
Batch 1 done
Monte Carlo pass 1/10 completed
Monte Carlo pass 2/10 completed
Monte Carlo pass 3/10 completed
Monte Carlo pass 4/10 completed
Monte Carlo pass 5/10 completed
Monte Carlo pass 6/10 completed
Monte Carlo pass 7/10 completed
Monte Carlo pass 8/10 completed
Monte Carlo pass 9/10 completed
Monte Carlo pass 10/10 completed
Batch 2 done
Monte Carlo pass 1/10 completed
Monte Carlo pass 2/10 completed
Monte Carlo pass 3/10 completed
Monte Carlo pass 4/10 completed
Monte Carlo pass 5/10 completed
Monte Carlo pass 6/10 completed
Monte Carlo pass 7/10 completed
Mo

# Swin-transformer Inference

In [13]:
test_torch_dataset = make_torch_dataset(x_test, y_test, training = False)

swin_model.eval()
all_predictions = []
all_labels = []

with torch.inference_mode():
    for batch_x, batch_y in test_torch_dataset:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        prediction = swin_model(batch_x)
        prediction = torch.sigmoid(prediction)
        all_predictions.append(prediction)
        all_labels.append(batch_y)

# Join sequence of tensors
all_predictions = torch.cat(all_predictions, dim = 0)
all_labels = torch.cat(all_labels, dim = 0)

# Swin-transformer Metrics

In [ ]:
swin_loss = metric_torch.bce_dice_loss(all_labels, all_predictions).item()
swin_dice = metric_torch.dice_coef(all_labels, all_predictions).item()
swin_hausdorffdistance = metric_torch.hausdorff_distance(all_labels, all_predictions)

print(f"Swin Loss: {swin_loss}")
print(f"Swin Dice: {swin_dice}")
print(f"Swin Hausdorff Distance: {swin_hausdorffdistance}")

# Swin MC Dropout
torch.cuda.empty_cache()

batch_size = 4
num_passes = 10
all_swin_means = []
all_swin_stds = []

for i in range(0, len(x_test), batch_size):
    batch_x = torch.from_numpy(x_test[i : i + batch_size]).permute(0, 4, 1, 2, 3).float().to(device)
    batch_predictions = torch.stack([torch.sigmoid(swin_model(batch_x)) for _ in range(num_passes)])

    all_swin_means.append(batch_predictions.mean(0).cpu().numpy())
    all_swin_stds.append(batch_predictions.std(0).cpu().numpy())

    del batch_x, batch_predictions
    torch.cuda.empty_cache()
    print(f"Batch {i // batch_size + 1} done")

swin_mean = np.concatenate(all_swin_means, axis = 0)
swin_std = np.concatenate(all_swin_stds, axis = 0)

print(f"Swin MC Uncertainty — Mean std: {swin_std.mean():.6f},  Max std: {swin_std.max():.6f}")
print(f"Swin MC Loss (on mean prediction): {metric_torch.bce_dice_loss(torch.from_numpy(y_test).permute(0,4,1,2,3).float(), torch.from_numpy(swin_mean)):.4f}")

Swin Loss: 0.6297786831855774
Swin Dice: 0.7649421095848083
Swin Hausdorff Distance: 20.86038507979887


# Plot Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt

with open("unet_history.json") as unet_file:
    unet_history = json.load(unet_file)
with open("swin_history.json") as swin_file:
    swin_history = json.load(swin_file)

epochs = range(1, len(unet_history["loss"]) + 1)

# U-Net Loss plot
plt.plot(epochs, unet_history["loss"], label = "Train")
plt.plot(epochs, unet_history["val_loss"], label = "Validation")
plt.title("U-Net Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig("plots/training_curves/unet_loss.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# U-Net Dice plot
plt.plot(epochs, unet_history["dice_coef"], label = "Train")
plt.plot(epochs, unet_history["val_dice_coef"], label = "Validation")
plt.title("U-Net Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice Coef")
plt.legend()
plt.savefig("plots/training_curves/unet_dice.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Swin Loss plot
plt.plot(epochs, swin_history["train_loss"], label = "Train")
plt.plot(epochs, swin_history["val_loss"], label = "Validation")
plt.title("Swin-UNETR Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig("plots/training_curves/swin_loss.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Swin Dice plot
plt.plot(epochs, swin_history["train_dice"], label = "Train")
plt.plot(epochs, swin_history["val_dice"], label = "Validation")
plt.title("Swin-UNETR Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice Coef")
plt.legend()
plt.savefig("plots/training_curves/swin_dice.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Plot Test Metrics Bar Chart

In [ ]:
models = ["U-Net", "Swin-UNETR"]
colors = ["darkblue", "darkorange"]

# Loss comparisons
losses = [unet_loss, swin_loss]
plt.bar(models, losses, color = colors)
plt.title("Binary Cross-Entropy + Dice Loss Comparison") # Lower is better
plt.ylim(0, 1)
plt.savefig("plots/metrics/bce_dice_comparison.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Dice Coefficients
coefficients = [unet_dice, swin_dice]
plt.bar(models, coefficients, color = colors)
plt.title("Dice Coefficient Comparison") # Higher is better
plt.ylim(0, 1)
plt.savefig("plots/metrics/dice_coef_comparison.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Hausdorff Distances
hausdorffdistances = [unet_hausdorffdistance, swin_hausdorffdistance]
plt.bar(models, hausdorffdistances, color = colors)
plt.title("Hausdorff Distance Comparison") # Higher is better
plt.ylim(0, 30)
plt.savefig("plots/metrics/hausdorffdistances_comparison.png", dpi = 1200, bbox_inches = 'tight')
plt.show()

# Plot Sample Predictions

In [ ]:
# Choose which samples and get the middle slice
sample_indices = [0, 1, 2]
mid_slice = x_test.shape[1] // 2

swin_predictions_np = all_predictions.cpu().numpy() # Currently looking like [N, 1, D, H, W]
swin_predictions_np = swin_predictions_np[:, 0, :, :, :] # [N, D, H, W] so it squeezes the channel dim

fig, axes = plt.subplots(3, 4, figsize = (16, 12))
fig.suptitle("Middle Slice Sample Predictions")
col_titles = ["Input (T1)", "Ground Truth", "U-Net Prediction", "Swin-UNETR Prediction"]

for row, idx in enumerate(sample_indices):
    # T1 slice
    input_slice = x_test[idx, mid_slice, :, :, 0]
    # Binary ground truth
    ground_truth_slice = y_test[idx, mid_slice, :, :, 0]
    # U-Net predicted probability map
    unet_slice = unet_predictions[idx, mid_slice, :, :, 0]
    # Swin predicted probability map
    swin_slice = swin_predictions_np[idx, mid_slice, :, :]

    axes[row, 0].imshow(input_slice, cmap="gray")
    axes[row, 1].imshow(ground_truth_slice, cmap="gray")
    axes[row, 2].imshow(unet_slice, cmap="gray")
    axes[row, 3].imshow(swin_slice, cmap="gray")

    for col in range(4):
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(col_titles[col])

plt.tight_layout()
plt.savefig("plots/samples/sample_predictions.png", dpi = 1200, bbox_inches = 'tight')
plt.show()